<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/09_sorter_v2_and_rating_sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [3]:
import re
from pathlib import Path

import jiwer
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

results = pd.read_csv(folder / "scores_200.csv", dtype={"file_id": str}, keep_default_na=False)
print(len(results), "rows")

Mounted at /content/drive
200 rows


In [4]:
digit_map = str.maketrans("०१२३४५६७८९", "0123456789")
vowel_signs = "ािीुूृेैोौ्ंँः"


def remove_punctuation(text):
    for mark in "।,.?!;:\"'":
        text = text.replace(mark, " ")
    return " ".join(text.split())


def has_latin_letters(text):
    return any("a" <= letter.lower() <= "z" for letter in text)


def count_broken_words(text):
    count = 0
    for word in text.split():
        if word[0] in vowel_signs:
            count += 1
    return count


def find_numbers(text):
    return re.findall(r"[0-9]+", text.translate(digit_map))

In [5]:
def edit_distance(first, second):
    previous_row = list(range(len(second) + 1))
    for i, letter_1 in enumerate(first, start=1):
        current_row = [i]
        for j, letter_2 in enumerate(second, start=1):
            cost = 0 if letter_1 == letter_2 else 1
            current_row.append(min(previous_row[j] + 1, current_row[j - 1] + 1, previous_row[j - 1] + cost))
        previous_row = current_row
    return previous_row[-1]


print(edit_distance("धेरै", "घेरै"))

1


In [6]:
def only_small_word_changes(correct, model):
    output = jiwer.process_words(correct, model)
    ref_words = output.references[0]
    hyp_words = output.hypotheses[0]

    for chunk in output.alignments[0]:
        if chunk.type in ["insert", "delete"]:
            return False
        if chunk.type == "substitute":
            for k in range(chunk.ref_end_idx - chunk.ref_start_idx):
                ref_word = ref_words[chunk.ref_start_idx + k]
                hyp_word = hyp_words[chunk.hyp_start_idx + k]
                if edit_distance(ref_word, hyp_word) > 2:
                    return False
    return True

In [7]:
def find_error_type(correct, model):
    if correct == model:
        return "exact"

    a = remove_punctuation(correct)
    b = remove_punctuation(model)
    if a == b:
        return "punctuation_only"
    if a.replace(" ", "") == b.replace(" ", ""):
        return "spacing_only"
    if b == "":
        return "big_difference"

    if count_broken_words(model) >= 2:
        return "junk"
    if has_latin_letters(model):
        return "latin_mix"

    a_digits = a.translate(digit_map)
    b_digits = b.translate(digit_map)
    if a_digits.replace(" ", "") == b_digits.replace(" ", ""):
        return "digit_script_only"
    if find_numbers(a) != find_numbers(b):
        return "number_wrong"

    if only_small_word_changes(a, b):
        return "small_difference"
    return "big_difference"

In [8]:
print(find_error_type("सबैकुरा सिकाउँथे", "सबै कुरा सिकाउँथे"))
print(find_error_type("आफ्नो पुस्तक २०३२", "आफ्नो पुस्तक 2032"))
print(find_error_type("कृपया यहाँ", "कृपeya yaha"))
print(find_error_type("गर्न छोड्नु।", "गर्न छोड्नु"))
print(find_error_type("१६ औँ शताब्दीमा", "16 ौँ शताब्दीमा"))
print(find_error_type("तीन वर्ष पछि", "3िन वर्ष पछि"))
print(find_error_type("पानीमा धेरै समय", "पानीमा घेरै समय"))

spacing_only
digit_script_only
latin_mix
punctuation_only
small_difference
number_wrong
small_difference


In [9]:
results["error_type"] = [find_error_type(c, m) for c, m in zip(results["correct_text"], results["model_text"])]
results["error_type"].value_counts()

,count
error_type,
exact,55
small_difference,48
big_difference,42
latin_mix,26
number_wrong,15
spacing_only,11
junk,2
digit_script_only,1


In [10]:
old_sorted = pd.read_csv(folder / "sorted_200.csv", dtype={"file_id": str}, keep_default_na=False)
old_types = old_sorted[["file_id", "error_type"]].rename(columns={"error_type": "old_type"})

compare = results[["file_id", "error_type"]].merge(old_types, on="file_id")
print("rows with a new type:", (compare["error_type"] != compare["old_type"]).sum())
pd.crosstab(compare["old_type"], compare["error_type"])

rows with a new type: 25


error_type,big_difference,digit_script_only,exact,junk,latin_mix,number_wrong,small_difference,spacing_only
old_type,,,,,,,,
big_difference,23,0,0,0,0,0,4,0
digit_script_only,0,1,0,0,0,0,0,0
exact,0,0,55,0,0,0,0,0
junk,3,0,0,2,1,0,1,0
latin_mix,0,0,0,0,25,0,0,0
number_wrong,1,0,0,0,0,15,0,0
small_difference,15,0,0,0,0,0,43,0
spacing_only,0,0,0,0,0,0,0,11


In [11]:
group_of = {
    "exact": "exact",
    "punctuation_only": "harmless",
    "spacing_only": "harmless",
    "digit_script_only": "harmless",
    "small_difference": "needs a human",
    "junk": "real mistake",
    "latin_mix": "real mistake",
    "number_wrong": "real mistake",
    "big_difference": "real mistake",
}
results["group"] = results["error_type"].map(group_of)
print(results["group"].value_counts())

results["word_errors"] = results["wer"] * results["correct_text"].str.split().str.len()
share = results.groupby("group")["word_errors"].sum() / results["word_errors"].sum()
print((share * 100).round(1))

group
real mistake     85
exact            55
needs a human    48
harmless         12
Name: count, dtype: int64
group
exact             0.0
harmless          6.9
needs a human    16.8
real mistake     76.3
Name: word_errors, dtype: float64


In [12]:
need_rating = results[results["group"].isin(["harmless", "needs a human"])]
control_rows = results[results["group"] == "real mistake"].sample(n=20, random_state=3)

rating_rows = pd.concat([need_rating, control_rows]).sample(frac=1, random_state=3).reset_index(drop=True)

rating_sheet_round2 = rating_rows[["file_id", "correct_text", "model_text"]]
rating_sheet_round2.to_csv(folder / "rating_sheet_round2.csv", index=False)
rating_rows.to_csv(folder / "rating_rows_private_round2.csv", index=False)

print(len(rating_sheet_round2), "rows to rate")
print(rating_rows["group"].value_counts())

80 rows to rate
group
needs a human    48
real mistake     20
harmless         12
Name: count, dtype: int64


In [13]:
results.to_csv(folder / "sorted_200_v2.csv", index=False)
print("saved")

saved


In [14]:
moved = compare[(compare["old_type"] == "small_difference") & (compare["error_type"] == "big_difference")]
moved_rows = results[results["file_id"].isin(moved["file_id"])]
moved_rows[["correct_text", "model_text"]].head(8)

,correct_text,model_text
4,सङ्घर्षका कार्यक्रमहरू पनि,संगर्षका कार्यक्रमहरू पनि
12,लिनक्स खुला श्रोत अन्तर्गतको युनिक्स अपरेटिङ स...,लिनङश खुलासोरोत अन्तर्गतको यउनिक्स अपरेटिङ सिस...
46,दक्षिण पूर्वी अस्ट्रेलियामा,डक्षिणपूर्वी अष्ट्रेलियामा
55,डलर पारिश्रमिक,दलर पारि श्रमिक
59,परिवार अन्तर्गत नेपार,परिवार अन्तर्घटनेपार
67,आक्रमण भै रहन्छ,आक्रमण भइरहन्छ
76,सशस्त्र विद्रोही समूहहरूको,सशस्त्र विद्रु ही समूहहरूको
82,सङ्गठितरूपमा लाग्नुभएको थियो,संगठित रूपमा लाग्नुभएको थियो
